In [1]:
import torch
import torchvision
from torch import nn
from torchvision import models
import rootutils
from torchvision.transforms import v2
import tqdm
import numpy as np
SEED = 42

In [2]:
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
rootutils.setup_root(".", indicator=".project-root", pythonpath=True)
from src.data.components.TransformSubset import TransformSubset

In [4]:
class EfficientNetBaseline(nn.Module):
    def __init__(self, num_classes: int, pretrained: bool = True) -> None:
        """Initialize EfficientNetBaseline.

        :param num_classes: Number of output classes.
        :param pretrained: Whether to use pretrained ImageNet weights. Defaults to True.
        """
        super().__init__()

        if pretrained:
            self.model = models.efficientnet_b0(
                weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
            )
        else:
            self.model = models.efficientnet_b0(weights=None)

        # Replace classifier head
        num_ftrs = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(num_ftrs, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

In [5]:
train_transforms = v2.Compose(
    [
        v2.Resize((640, 480), antialias=True),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

In [6]:
device = torch.device("cuda", index=0)
torch.set_float32_matmul_precision("high")

scaler = torch.amp.GradScaler('cuda')

In [7]:
data_path = "../data/surface/train"
dataset = torchvision.datasets.ImageFolder(root=data_path)

class_counts = np.unique(dataset.targets, return_counts=True)[1]
weights = 1.0 / torch.tensor(class_counts, dtype=torch.float32)
weights = weights / weights.sum()

In [8]:
from torchmetrics import MeanMetric

/home/lukasb/miniforge3/envs/defect-detection/lib/python3.10/site-packages/torchmetrics/utilities/imports.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [9]:
# Training parameters
BATCH_SIZE = 32
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-5

In [10]:
import os

# AUM/AUM-cleaning configuration
UPPER_PERCENT = 10  # bottom X% by AUM to mark as clean
FLAGGED_DIR = os.path.join("../data/surface/clean_test")
CREATE_CLASS_SUBFOLDERS = True

In [11]:
# Wrap dataset so we can get sample indices and file paths for AUM logging
from PIL import Image

class IndexDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, transform):
        self.base = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
       path, label = self.base.samples[idx]
       img = Image.open(path).convert('RGB')
       img = self.transform(img)
       return img, label, idx, path

index_dataset = IndexDataset(dataset, train_transforms)

In [12]:
train_dataloader = torch.utils.data.DataLoader(
    index_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
)

In [13]:
num_classes = len(dataset.classes)
model = EfficientNetBaseline(num_classes=num_classes, pretrained=True)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=weights.to(device))
criterion = criterion.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

train_loss = MeanMetric().to(device)

In [14]:
# Helper to compute per-sample margins from logits and labels (on the same device)
def compute_margins(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    # correct-class logit
    correct = logits.gather(1, labels.view(-1, 1)).squeeze(1)
    # top-2 logits; pick the competitor (if top-1 is correct, use top-2, else use top-1)
    top2_vals, top2_idx = logits.topk(k=2, dim=1)
    competitor = torch.where(top2_idx[:, 0].eq(labels), top2_vals[:, 1], top2_vals[:, 0])
    return correct - competitor

In [15]:
aum_sum = torch.zeros(len(index_dataset), dtype=torch.float64)
aum_cnt = torch.zeros(len(index_dataset), dtype=torch.int32)

for epoch in range(EPOCHS):
    model.train()
    train_loss.reset()

    # Add leave=False to clear progress bar after completion
    pbar = tqdm.tqdm(train_dataloader, desc=f"Epoch {epoch}", leave=False, position=0)
    for images, labels, idx, path in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        # Mixed precision training
        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        scheduler.step()

        with torch.no_grad():
            preds = torch.argmax(logits, dim=1)
            train_loss.update(loss.item())

        with torch.no_grad():
            margins = compute_margins(logits, labels).detach().cpu().to(torch.float64)
            aum_sum[idx] += margins
            aum_cnt[idx] += 1
        
print(f"Epoch {epoch} - Train Loss: {train_loss.compute():.4f}")

KeyboardInterrupt: 

In [16]:
aum_scores = (aum_sum / aum_cnt.clamp_min(1)).numpy()

paths = []
labels = []

for img, label, idx, path in index_dataset:
    paths.append(path)
    labels.append(label)

import pandas as pd
clean_df = pd.DataFrame({
    'idx': np.arange(len(index_dataset)),
    'aum': aum_scores,
    'label': labels,
    'path': paths
})

In [19]:
per_class_df = [clean_df[clean_df['label'] == class_idx] for class_idx in range(num_classes)]

In [20]:
flag_df = pd.DataFrame(columns=clean_df.columns)

for class_idx, class_df in enumerate(per_class_df):
    cutoff = np.percentile(class_df['aum'], 100.0-15.0)
    class_flag_df = class_df[class_df['aum'] >= cutoff]
    flag_df = pd.concat([flag_df, class_flag_df])
    
print(f"Flagging {len(flag_df)} of {len(clean_df)} samples (top {15}%)")
flag_df

Flagging 844 of 5588 samples (top 15%)


/tmp/ipykernel_1875668/2712985859.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  flag_df = pd.concat([flag_df, class_flag_df])


,idx,aum,label,path
34,34,8.171655,0,../data/surface/train/Bent/20240617-224014_7.jpg
54,54,8.201819,0,../data/surface/train/Bent/20250513-143501_2.jpg
57,57,8.688251,0,../data/surface/train/Bent/20250513-143507_5.jpg
58,58,8.218677,0,../data/surface/train/Bent/20250513-143509_6.jpg
64,64,8.200145,0,../data/surface/train/Bent/20250513-143541_2.jpg
...,...,...,...,...
5450,5450,9.804590,11,../data/surface/train/Waterstains/20250612-113...
5462,5462,9.837598,11,../data/surface/train/Waterstains/20250612-113...
5500,5500,9.828857,11,../data/surface/train/Waterstains/20250630-095...
5502,5502,9.434009,11,../data/surface/train/Waterstains/20250630-095...


In [21]:
print(FLAGGED_DIR)
# Move flagged files for inspection
os.makedirs(FLAGGED_DIR, exist_ok=True)


../data/surface/clean_test


In [22]:
import shutil

class_names = dataset.classes
moved = 0
for _, row in flag_df.iterrows():
    src = row['path']
    cls_name = class_names[row['label']] if row['label'] is not None else 'unknown'
    if CREATE_CLASS_SUBFOLDERS:
        dst_dir = os.path.join(FLAGGED_DIR, cls_name)
        os.makedirs(dst_dir, exist_ok=True)
        dst = os.path.join(dst_dir, os.path.basename(src))
    else:
        dst = os.path.join(FLAGGED_DIR, f"{cls_name}___{os.path.basename(src)}")
    try:
        if os.path.exists(src):
            shutil.copy2(src, dst)
            moved += 1
    except Exception as e:
        print(f"Failed to copy {src} -> {dst}: {e}")

print(f"Copied {moved} flagged samples to {FLAGGED_DIR}")

Copied 844 flagged samples to ../data/surface/clean_test
